In [86]:
# Using -q for a quieter installation in the notebook
!pip install datasets tiktoken tqdm numpy matplotlib pandas -q

# --- Core Torch and Helpers ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass
from torch import tensor

# --- Data Handling ---
import numpy as np
from datasets import load_dataset
import tiktoken
from tqdm.auto import tqdm

# --- Plotting and Analysis ---
import matplotlib.pyplot as plt
import pandas as pd

print("All dependencies installed and imported successfully.")
# Verify GPU is available
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected. Running on CPU.")


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
All dependencies installed and imported successfully.
GPU detected: NVIDIA GeForce RTX 5090
PyTorch version: 2.8.0.dev20250319+cu128
CUDA version: 12.8


In [87]:
# --- Global Training Config ---
BATCH_SIZE = 32          # Number of sequences in a batch
BLOCK_SIZE = 256         # Context length of the model
MAX_ITERS = 5000         # Total training iterations
EVAL_INTERVAL = 250      # How often to evaluate and print loss
EVAL_ITERS = 100         # Number of batches to average for validation loss
LEARNING_RATE = 3e-4     # Max learning rate for the optimizer
WARMUP_ITERS = 200       # Number of iterations for learning rate warmup
MIN_LR = 3e-5            # Minimum learning rate after cosine decay

In [88]:
# --- Device and Precision Config ---
# Automatically selects 'cuda' if available, otherwise 'cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Use bfloat16 for training if available (on A100+ GPUs), otherwise float16.
# This maximizes performance and prevents common float16 overflow issues.
DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
PTDTYPE = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}[DTYPE]
# Create the autocast context manager for mixed-precision training
CTX = nullcontext() if DEVICE == 'cpu' else torch.amp.autocast(device_type=DEVICE, dtype=PTDTYPE)
# GradScaler is only needed for float16, not bfloat16 or float32
SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))

/tmp/ipykernel_102/3088237724.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler(enabled=(DTYPE == 'float16'))


In [ ]:
# --- Base Model Architecture Config ---
# We will use these settings for all four models to ensure a fair comparison.
@dataclass
class BaseModelConfig:
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 256
    vocab_size: int = 50257  # Standard for GPT-2 tokenizer
    block_size: int = BLOCK_SIZE
    dropout: float = 0.1
    bias: bool = False # Set to False for LayerNorms and Linears, as is common practice
    
    MLA_COMPRESSION_RATIO = 0.5
    d_latent: int = int(256 * MLA_COMPRESSION_RATIO) # d_model * ratio
    d_rope: int = 16 

In [90]:
# Create an instance of the config
BASE_CONFIG = BaseModelConfig()

In [91]:
GQA_N_GROUPS = 2
assert BASE_CONFIG.n_head % GQA_N_GROUPS == 0, "n_head must be divisible by n_groups for GQA"

In [92]:
print(f"Running on device: {DEVICE}")
print(f"Using data type: {DTYPE}")
print(f"Base model config: {BASE_CONFIG.n_layer} layers, {BASE_CONFIG.n_head} heads, {BASE_CONFIG.n_embd} embedding dim.")

Running on device: cuda
Using data type: bfloat16
Base model config: 6 layers, 8 heads, 256 embedding dim.


In [93]:
# --- Dataset Configuration ---
DATASET_NAME = "roneneldan/TinyStories"
TRAIN_BIN_FILE = "train.bin"
VAL_BIN_FILE = "val.bin"
TOKENIZER_MODEL = "gpt2"

In [94]:
# --- Tokenization and File Writing ---
if not (os.path.exists(TRAIN_BIN_FILE) and os.path.exists(VAL_BIN_FILE)):
    print("Binary data files not found. Starting data download and tokenization...")

    # Load the dataset from Hugging Face
    ds = load_dataset(DATASET_NAME)

    # Initialize the tokenizer
    enc = tiktoken.get_encoding(TOKENIZER_MODEL)
    assert enc.max_token_value < 2**16, "Tokenizer vocab size is too large for uint16"

    def process_and_tokenize(example):
        """Tokenize a single text example and return token IDs and length."""
        ids = enc.encode_ordinary(example['text'])
        return {'ids': ids, 'len': len(ids)}

    # Tokenize the dataset in parallel
    tokenized_ds = ds.map(
        process_and_tokenize,
        remove_columns=['text'],
        desc="Tokenizing splits",
        num_proc=os.cpu_count(),
    )

    # Write tokenized data to memory-mapped binary files
    for split, dset in tokenized_ds.items():
        # Use a consistent naming convention (validation -> val)
        split_name = 'val' if split == 'validation' else split
        filename = f"{split_name}.bin"

        total_tokens = np.sum(dset['len'], dtype=np.uint64)
        print(f"Found {total_tokens:,} tokens in the '{split}' split.")

        # Use uint16 for memory efficiency, as vocab size allows it
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(total_tokens,))

        print(f"Writing tokens to {filename}...")
        current_idx = 0
        # Process in batches to manage memory during concatenation
        for batch in tqdm(dset.iter(batch_size=2048), total=len(dset)//2048):
            concatenated_ids = np.concatenate(batch['ids'])
            arr[current_idx : current_idx + len(concatenated_ids)] = concatenated_ids
            current_idx += len(concatenated_ids)

        arr.flush()
    print("Tokenization and file writing complete.")
else:
    print("Found existing train.bin and val.bin files. Skipping data preparation.")

Found existing train.bin and val.bin files. Skipping data preparation.
